In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

avg_ratings = ratings.groupby("movieId")["rating"].mean().reset_index()
avg_ratings.rename(
    columns={"rating": "avg_rating"},
    inplace=True
)
movies = movies.merge(
    avg_ratings,
    on="movieId",
    how="left"
)
movies["avg_rating"] = movies["avg_rating"].fillna(0)
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(
    movies["genres"]
)

cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

def recommend(movie_title, n=10):
    if movie_title not in indices:
        print("Movie not found.")
        return
    idx = indices[movie_title]
    similarity_scores = list(
        enumerate(
            cosine_sim[idx]
        )
    )
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    similarity_scores = similarity_scores[1:50]
    movie_indices = [
        i[0]
        for i in similarity_scores
    ]
    recommendations = movies.iloc[
        movie_indices
    ][
        [
            "title",
            "genres",
            "avg_rating"
        ]
    ]

    recommendations = recommendations[
        recommendations["avg_rating"] >= 3.5
    ]

    recommendations = recommendations.sort_values(
        by="avg_rating",
        ascending=False
    )
    return recommendations.head(n)

while True:

    movie_name = input(
        "\nEnter Movie Name (or quit): "
    )

    if movie_name.lower() == "quit":
        break

    result = recommend(movie_name)

    if result is not None:
        print("\nRecommended Movies:\n")
        print(result[["title", "avg_rating"]])



Enter Movie Name (or quit): Thor Ragnarok
Movie not found.

Enter Movie Name (or quit): Inception
Movie not found.

Enter Movie Name (or quit): Interstellar (2014)

Recommended Movies:

                                         title  avg_rating
13684                         Star Trek (2009)    3.986486
10886                    V for Vendetta (2006)    3.945068
17874                     Avengers, The (2012)    3.933661
20994           Star Trek Into Darkness (2013)    3.818452
14592                            Avatar (2009)    3.817272
21646                           Gravity (2013)    3.750000
22115  Hunger Games: Catching Fire, The (2013)    3.733974
4359                               More (1998)    3.718750
20906                        Iron Man 3 (2013)    3.689542
23475                  Edge of Tomorrow (2014)    3.676230

Enter Movie Name (or quit): quit
